In [11]:
import json
import requests
from openai import OpenAI

client = OpenAI()


def get_popular_movies():
    url = "https://nomad-movies.nomadcoders.workers.dev/movies"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()


def get_movie_details(id):
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()


def get_movie_credits(id):
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}/credits"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()

def get_movie_videos(id):
    # get movide videos by id   
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}/videos"
    resp = requests.get(url, timeout= 10)
    resp.raise_for_status()
    return resp.json()


TOOLS = [
    {
        "type": "function",
        "name": "get_popular_movies",
        "description": "Get a list of currently popular movies.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_movie_details",
        "description": "Get detailed information about a movie by id.",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "The movie id"
                }
            },
            "required": ["id"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_movie_credits",
        "description": "Get cast and crew credits for a movie by id.",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "The movie id"
                }
            },
            "required": ["id"],
            "additionalProperties": False,
        },
    },
    {
        "type":"function",
        "name":"get_movie_videos",
        "description": "Get videos that is related with movie by id",
        "parameters": {
            "type":"object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "the movie id"
                },
            },
            "required":["id"],
            "additionalProperties": False
        }
    }
]


def call_tool(tool_name, args):
    if tool_name == "get_popular_movies":
        return get_popular_movies()
    elif tool_name == "get_movie_details":
        return get_movie_details(args["id"])
    elif tool_name == "get_movie_credits":
        return get_movie_credits(args["id"])
    elif tool_name == "get_movie_videos":
        return get_movie_videos(args["id"])
    else:
        raise ValueError(f"Unknown tool: {tool_name}")


def run_agent(user_input):
    response = client.responses.create(
        model="gpt-4o-mini",
        input=user_input,
        instructions=("You are a helpful movie assistant. "
        "Use the provided tools whenever the user asks for real movie data. " 
        "If the user asks for popular movies, use get_popular_movies and include each movie id next to the title.  " 
        "If the user asks for details, use get_movie_details. " 
        "If the user asks for cast or crew, use get_movie_credits. "
        "If the user asks for trailers or videos, use get_movie_videos. "
        "Answer clearly in Korean."
        ),
        tools=TOOLS,
    )

    function_calls = [
        item for item in response.output
        if item.type == "function_call"
    ]

    if not function_calls:
        return response.output_text

    call = function_calls[0]

    tool_name = call.name
    args = json.loads(call.arguments)

    print(f"🔧 Function Call:, {tool_name}, and the ID is {args}")

    tool_result = call_tool(tool_name, args)

    final_response = client.responses.create(
        model="gpt-4o-mini",
        previous_response_id=response.id,
        input=[{
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": json.dumps(tool_result, ensure_ascii=False),
        }],
    )

    return final_response.output_text


In [9]:

result = run_agent("지금 인기 있는 영화 2개만 알려줘")
print(result)


🔧 Function Call:, get_popular_movies, and the ID is {}
현재 인기 있는 영화 두 편을 소개할게요.

1. **제목**: [Your Heart Will Be Broken](https://image.tmdb.org/t/p/w780/7wIBfBl2gejt6xHxNSK0reVIm7E.jpg)
   - **개요**: 고등학생 폴리나는 새 학교에서 괴롭힘을 당하다가 주된 괴롭힘을 주는 학생인 바르스와 연애하는 척하는 계약을 맺습니다. 이 게임을 하는 동안 두 사람은 진정한 감정을 발전시키지만, 가족과 동급생들은 그들을 갈라놓으려 합니다.
   - **평점**: 6.9
   - **개봉일**: 2026-03-26

2. **제목**: [Avatar: Fire and Ash](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)
   - **개요**: RDA와의 치명적인 전쟁 후, 제이크 술리와 네이티리는 새로운 위협인 잿더미 부족과 맞서 싸워야 합니다. 그들은 생존과 판도라의 미래를 위해 싸워야 하며, 이는 그들의 감정적, 신체적 한계를 시험하게 됩니다.
   - **평점**: 7.4
   - **개봉일**: 2025-12-17

두 영화 모두 기대할 만한 줄거리와 매력을 가지고 있어요!


In [10]:

result = run_agent("아바타에 대해서 더알려줘")
print(result)


🔧 Function Call:, get_popular_movies, and the ID is {}
"아바타(Avatar)"는 제임스 카메론 감독의 2009년 개봉한 영화로, 놀라운 시각 효과와 혁신적인 3D 기술로 유명합니다. 

### 기본 정보
- **제목**: 아바타
- **장르**: 액션, 어드벤처, 공상과학
- **줄거리**: 지구가 자원을 고갈하면서 인류는 판도라라는 외계 행성을 탐사하게 됩니다. 이곳의 원주민인 나비(Na'vi)와의 갈등 속에서 주인공 제이크 설리(Jake Sully)는 나비의 아바타로서 그들의 문화를 경험하며 그들과의 연대와 갈등을 겪게 됩니다.

### 주요 테마
1. **환경 보호**: 영화는 자연과의 조화, 그리고 자원의 착취라는 주제를 다룹니다.
2. **식민지적 테마**: 인간의 탐욕과 다른 종족에 대한 침략의 문제를 비판합니다.
3. **정체성**: 제이크 설리는 자신의 정체성을 찾고, 두 세계 사이에서 결정해야 하는 dilemmas을 겪습니다.

### 시리즈
아바타는 첫 번째 영화의 성공 후 여러 후속편이 계획되고 있으며, 최신 소식으로는 "Avatar: Fire and Ash"가 2025년에 출시될 예정입니다. 이 후속편에서는 제이크 설리와 네이티리의 새로운 도전이 다뤄질 예정입니다.

### 기술 혁신
아바타는 당시 최고의 CGI 기술을 사용하여 판도라를 사실감 있게 재현했으며, 3D 영화 경험의 새로운 기준을 설정했습니다.

영화의 성공으로 인해 아바타는 문화와 영화 역사에 큰 영향을 미쳤으며, 다양한 팬덤과 비평가들의 찬사를 받았습니다.


In [26]:

result = run_agent("movie ID 550에 해당하는 영화가 무엇인지 알려줘")
print(result)


🔧 Function Call: get_movie_details {'id': 550}
영화 ID 550에 해당하는 영화는 **"Fight Club"**입니다. 

### 기본 정보:
- **개봉일**: 1999년 10월 15일
- **장르**: 드라마, 스릴러
- **러닝타임**: 139분
- **예산**: $63,000,000
- **수익**: $100,853,753
- **평점**: 8.4 (투표 수: 31,729)

### 줄거리:
잠든 듯한 삶을 사는 한 남자와 교묘한 비누 판매자가 본능적인 남성성을 새로운 형태의 치료로 채널링하는 이야기를 다룹니다. 이들의 개념은 각 도시에서 언더그라운드 "격투 클럽"으로 번져 나가지만, 한 괴짜가 끼어들면서 혼돈의 나락으로 빠져들게 됩니다.

### 제작사:
- Fox 2000 Pictures
- Regency Enterprises
- Linson Entertainment
- 20th Century Fox
- Taurus Film

### 포스터:
![Fight Club Poster](https://image.tmdb.org/t/p/w780/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg)

### 더 알아보기:
[Fight Club 홈페이지](https://www.20thcenturystudios.com/movies/fight-club)

궁금한 점이 있다면 말씀해 주세요!
